# Embed the Summaries

| Item | Description |
|---|---|
| Goal | Embed every filing summary (all prompt versions in the table) and every firm summary with the same encoder and pooling as the topic vectors and the Item 1 sentences, so T3.8.1 can take cosines between them. |
| Input | `<scratch>/T3.7 - Firm Summaries/Processed/filing_summaries.parquet` (T3.7.3) and `firm_summaries.parquet` (T3.7.5). |
| Output | `<scratch>/T3.7 - Firm Summaries/Processed/filing_summary_embeddings.npy` (n × 1024, float32) with `filing_summary_embedding_keys.parquet` (`accessionNo`, `family`, `prompt_version`) in the same row order; `firm_summary_embeddings.npy` with `firm_summary_embedding_keys.parquet` (`gvkey`, `set`, `family`, `prompt_version`); `output/T3.7.6/embedding_record.json`. |
| Unit | One summary, one vector. |
| Model | `BAAI/bge-large-en-v1.5`, CLS token of the last layer, L2-normalized, inputs at most 512 tokens (every summary is within the limit by construction; T3.7.3 and T3.7.5 assert it). The same settings as T3.5.1, T3.5.3, T3.5.7, and T3.6.3. |

Run on a GPU node. See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer


## Step 2. Define file locations and settings


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.7.6"  # the identifier names this notebook's output folder
PROCESSED_FOLDER = SCRATCH_FOLDER / "T3.7 - Firm Summaries" / "Processed"
FILING_TABLE_FILE = PROCESSED_FOLDER / "filing_summaries.parquet"
FIRM_TABLE_FILE = PROCESSED_FOLDER / "firm_summaries.parquet"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME
MODEL_FOLDER = Path(
    "/scratch2/acroh/projects/llm_method/hf_cache/hub/models--BAAI--bge-large-en-v1.5"
    "/snapshots/d4aa6901d3a41ba39fb536a557fa166f842b0e09"
)

MAX_TOKENS = 512  # the encoder's input limit
BATCH_SIZE = 64  # summaries per forward pass
FILING_KEYS = ["accessionNo", "family", "prompt_version"]
FIRM_KEYS = ["gvkey", "set", "family", "prompt_version"]
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "run this notebook on a GPU node"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Outputs: {PROCESSED_FOLDER} and {OUTPUT_FOLDER}")


GPU: NVIDIA A40
Outputs: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.7 - Firm Summaries/Processed and /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.6


## Step 3. Load the two summary tables

Every row of each table gets a vector; the key columns identify the row.


In [3]:
filing_table = pd.read_parquet(FILING_TABLE_FILE)
firm_table = pd.read_parquet(FIRM_TABLE_FILE)
assert not filing_table.duplicated(FILING_KEYS).any() and not firm_table.duplicated(FIRM_KEYS).any()
assert filing_table["n_bge_tokens"].le(MAX_TOKENS).all() and firm_table["n_bge_tokens"].le(MAX_TOKENS).all()
print(f"Filing summaries: {len(filing_table):,} rows")
print(filing_table.groupby(["set", "family", "prompt_version"]).size().rename("rows").to_string())
print(f"Firm summaries: {len(firm_table):,} rows")
print(firm_table.groupby(["set", "family", "prompt_version"]).size().rename("rows").to_string())


Filing summaries: 22,923 rows
set          family                     prompt_version     
remaining    summary_lines_of_business  item1-mapreduce-1.3    16707
three_years  summary_lines_of_business  item1-mapreduce-1.1     2072
                                        item1-mapreduce-1.2     2072
                                        item1-mapreduce-1.3     2072
Firm summaries: 1,899 rows
set          family                     prompt_version     
all_years    summary_lines_of_business  item1-mapreduce-1.3    1058
three_years  summary_lines_of_business  item1-mapreduce-1.3     841


## Step 4. Load the encoder


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_FOLDER, local_files_only=True)
model = AutoModel.from_pretrained(MODEL_FOLDER, local_files_only=True).to("cuda").eval()
print(f"Model: {MODEL_FOLDER.parent.parent.name}; hidden size {model.config.hidden_size}; dtype {next(model.parameters()).dtype}")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model: models--BAAI--bge-large-en-v1.5; hidden size 1024; dtype torch.float32


## Step 5. Embed the texts

Texts are sorted by length so each batch pads little, embedded as the CLS token of the last layer at unit length, and put back in table order.


In [5]:
def embed_texts(texts):
    texts = list(texts)
    order = np.argsort([len(text) for text in texts])
    vectors_sorted = np.empty((len(texts), model.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(texts), BATCH_SIZE):
            batch = [texts[i] for i in order[start:start + BATCH_SIZE]]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=MAX_TOKENS, return_tensors="pt").to("cuda")
            last_layer = model(**encoded).last_hidden_state
            cls_vectors = torch.nn.functional.normalize(last_layer[:, 0], dim=-1)
            vectors_sorted[start:start + len(batch)] = cls_vectors.cpu().numpy()
    vectors = np.empty_like(vectors_sorted)
    vectors[order] = vectors_sorted
    return vectors


started = datetime.now(timezone.utc)
filing_vectors = embed_texts(filing_table["summary"])
firm_vectors = embed_texts(firm_table["summary"])
seconds = (datetime.now(timezone.utc) - started).total_seconds()
assert filing_vectors.shape == (len(filing_table), model.config.hidden_size) and firm_vectors.shape == (len(firm_table), model.config.hidden_size)
assert np.allclose(np.linalg.norm(filing_vectors, axis=1), 1, atol=1e-4) and np.allclose(np.linalg.norm(firm_vectors, axis=1), 1, atol=1e-4)
print(f"Embedded {len(filing_vectors):,} filing summaries and {len(firm_vectors):,} firm summaries in {seconds / 60:.1f} minutes")


Embedded 22,923 filing summaries and 1,899 firm summaries in 5.5 minutes


## Step 6. Check the vectors

Re-embed a few summaries one at a time and compare with the batched vectors; then two sanity cosines: a firm summary against its own yearly summaries (should be high), and the same filing's summary across prompt versions (should be high but below one).


In [6]:
rng = np.random.default_rng(0)
check_rows = rng.choice(len(filing_table), size=5, replace=False)
single = np.vstack([embed_texts([filing_table["summary"].iloc[i]]) for i in check_rows])
print(f"Batched vs single-text cosine on 5 filing summaries: min {np.min(np.sum(single * filing_vectors[check_rows], axis=1)):.5f}")

firm_index = {tuple(key): i for i, key in enumerate(firm_table[FIRM_KEYS].itertuples(index=False, name=None))}
own = []
for i, row in enumerate(filing_table.itertuples(index=False)):
    j = firm_index.get((row.gvkey, row.set, row.family, row.prompt_version))
    if j is not None:
        own.append(float(filing_vectors[i] @ firm_vectors[j]))
own = pd.Series(own)
print(f"Cosine of a yearly summary with its firm summary ({len(own):,} pairs, same version): median {own.median():.3f}, 5th percentile {own.quantile(0.05):.3f}")

versions = sorted(filing_table["prompt_version"].unique())
if len(versions) > 1:
    pivot = {v: filing_table.index[filing_table["prompt_version"].eq(v)] for v in versions}
    by_acc = {v: pd.Series(pivot[v], index=filing_table.loc[pivot[v], "accessionNo"]) for v in versions}
    for a in range(len(versions)):
        for b in range(a + 1, len(versions)):
            common = by_acc[versions[a]].index.intersection(by_acc[versions[b]].index)
            cos = np.sum(filing_vectors[by_acc[versions[a]][common]] * filing_vectors[by_acc[versions[b]][common]], axis=1)
            print(f"Same filing, {versions[a]} vs {versions[b]} ({len(common):,} filings): median cosine {np.median(cos):.3f}")


Batched vs single-text cosine on 5 filing summaries: min 1.00000
Cosine of a yearly summary with its firm summary (2,072 pairs, same version): median 0.937, 5th percentile 0.873
Same filing, item1-mapreduce-1.1 vs item1-mapreduce-1.2 (2,072 filings): median cosine 0.816
Same filing, item1-mapreduce-1.1 vs item1-mapreduce-1.3 (2,072 filings): median cosine 0.811
Same filing, item1-mapreduce-1.2 vs item1-mapreduce-1.3 (2,072 filings): median cosine 0.960


## Step 7. Save the vectors, their keys, and the record


In [7]:
np.save(PROCESSED_FOLDER / "filing_summary_embeddings.npy", filing_vectors)
filing_table[FILING_KEYS].to_parquet(PROCESSED_FOLDER / "filing_summary_embedding_keys.parquet", index=False)
np.save(PROCESSED_FOLDER / "firm_summary_embeddings.npy", firm_vectors)
firm_table[FIRM_KEYS].to_parquet(PROCESSED_FOLDER / "firm_summary_embedding_keys.parquet", index=False)
record = {
    "model": str(MODEL_FOLDER), "pooling": "cls", "normalized": True, "max_tokens": MAX_TOKENS, "batch_size": BATCH_SIZE,
    "saved_dtype": "float32", "n_filing_summaries": int(len(filing_table)), "n_firm_summaries": int(len(firm_table)),
    "filing_versions": {f"{s}|{f}|{v}": int(n) for (s, f, v), n in filing_table.groupby(["set", "family", "prompt_version"]).size().items()},
    "gpu": torch.cuda.get_device_name(0), "seconds": round(seconds, 1), "finished": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(OUTPUT_FOLDER / "embedding_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print(f"Saved: {PROCESSED_FOLDER / 'filing_summary_embeddings.npy'} {filing_vectors.shape} and its keys")
print(f"Saved: {PROCESSED_FOLDER / 'firm_summary_embeddings.npy'} {firm_vectors.shape} and its keys")
print(f"Saved: {OUTPUT_FOLDER / 'embedding_record.json'}")


Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.7 - Firm Summaries/Processed/filing_summary_embeddings.npy (22923, 1024) and its keys
Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.7 - Firm Summaries/Processed/firm_summary_embeddings.npy (1899, 1024) and its keys
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.6/embedding_record.json
